This gets all the software we'll need into your working environment.

In [ ]:
import json
import piplite
import pyodide.http
import zipfile
from math import sqrt

from tqdm import tqdm
import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pyarrow as pa
import shapely

import data

await piplite.install("folium")
await piplite.install("mapclassify")

<br><br><br><br><br>

# 1. Introduction

_(Jim gives a presentation.)_

<br><br><br><br><br>

# 2. Gerrymandering challenge

You've been appointed by the Supreme Chancellor to draw the boundaries that define the galaxy's 9 sectors. Each planet in a sector gets one vote to elect a senator for its sector: 9 senators total. The Chancellor's Purple Party is popular in 200 planets; the opposing party is popular in 250 planets. Even though the Purple Party is less popular overall, draw the boundaries in a way to ensure a majority, at least 5 of the 9 senators, for the Purple Party.

I've managed to get as many as 7 Purple Party senators. Is it possible to get 8?

<a href="/2025-07-12-scipy-teen-track/gerrymandering-galaxy.svg"><img src="/2025-07-12-scipy-teen-track/img/gerrymandering-galaxy.svg" width="500"></a>

<br><br><br><br><br>

# 3. Jupyter practice

## Running Python code in Jupyter cells

You can run a cell by clicking on it and typing "control-enter" or "command-enter" (Mac).

You can run a cell and move to the next one with "shift-enter".

In [ ]:
# run this cell to find out what the answer is

2 + 3

In [ ]:
# now run this one

2 * 3

In [ ]:
# and now this one

2**3

## Running cells in order

The order in which you run cells matters!

In [ ]:
# step 4
x = x**2

In [ ]:
# step 2
x = x * 2

In [ ]:
# step 1
x = 1

In [ ]:
# step 5
x -= 20

In [ ]:
# step 3
x += 5

If you ran the steps in the right order, the value of `x` should be

```
29
```

In [ ]:
x

Note: you can rearrange the order of the cells by dragging them. Then it's easy to "shift-enter" through them.

It's good practice to set up a notebook so that you can run it from top to bottom. That way, you don't have to remember which cells are "supposed to" be run before others.

## Creating cells

Challenge: create 3 new cells below this one, put some code in them, and run them.

## If you need to start over

If you're not sure which cells have been run, in what order, and need to start over, you can "restart kernel".

<img src="/2025-07-12-scipy-teen-track/img/restart-kernel.png">

## Last question

Did you run the first cell with all of the "import" statements?

In [ ]:
sqrt

In [ ]:
plt

In [ ]:
data

<br><br><br><br><br>

# 4. What is a "fair" geometry?

## Polsby–Popper score

$$ \mbox{score} = 4\pi \frac{\mbox{area}}{\mbox{perimeter}^2} $$

> It is beyond the scope of this essay, to say nothing of its authors, to say whether a particular district ought to be compact or how compact it ought to be. But we can answer an easier question: whether a compactness criterion complicates the business of gerrymandering. It does. The third criterion will make the gerrymanderer's life a living hell. That's why we're for it.

Polsby, Daniel D.; Popper, Robert D. <a href="https://openyls.law.yale.edu/handle/20.500.13051/17448">"The Third Criterion: Compactness as a procedural safeguard against partisan gerrymandering"</a>. Yale Law & Policy Review. 9 (2): 301–353 (1991).

In [ ]:
#                           x      y
sample_shape = np.array([[ 2.68, 13.54],
                         [-5.8 , 14.15],
                         [-3.45,  7.62],
                         [ 7.99,  9.05],
                         [ 5.74,  1.59],
                         [20.05,  3.23],
                         [21.99,  9.25],
                         [17.49, 13.54],
                         [26.58, 24.26],
                         [12.08, 25.59],
                         [12.59, 17.01],
                         [ 2.88, 19.98],
                         [ 2.68, 13.54]])

In [ ]:
fig, ax = plt.subplots()

#       all x values        all y values
ax.plot(sample_shape[:, 0], sample_shape[:, 1], marker="o")
ax.set_xlabel("x")
ax.set_ylabel("y")

None

### Calculating perimeter

The length of a line segment that spans $\Delta x$ and $\Delta y$ is $\sqrt{\Delta x^2 + \Delta y^2}$.

The perimeter of a polygon is the sum of the lengths of its line segments.

<img src="/2025-07-12-scipy-teen-track/img/polygon-perimeter.svg" width="500">

Remove the `#` before `result` and replace the `???` with code to calculate the perimeter.

In [ ]:
def perimeter(polygon):
    result = 0

    for i in range(len(polygon) - 1):
        x_this, y_this = polygon[i]
        x_next, y_next = polygon[i + 1]
        # print(x_this, y_this, x_next, y_next)

        # result += ???

    return result

perimeter(sample_shape)

The resulting perimeter should be

```
115.51693885291068
```

### Calculating area

A trapezoid can be broken down into a rectangle and one or two right triangles.

Thea area of a rectangle is $\Delta x \Delta y$.

The area of a right triangle is $\frac{1}{2} \Delta x \Delta y$.

<img src="/2025-07-12-scipy-teen-track/img/trapezoid-area.svg" width="500">

A polygon can be broken up into trapezoids that all have a rectangular base on the $x$ axis and a triangular top. For each of these trapezoids, we can compute a "signed area," which is positive and equal to its area if $x_1 < x_2$ and equal to its negated area if $x_1 > x_2$. A complete polygon will always have both positive and negative pieces.

The area of the polygon is equal to the sum of signed areas of the trapezoids. Some trapezoids include area outside of the polygon, but it's exactly cancelled out by polygons with negative signed area.

<img src="/2025-07-12-scipy-teen-track/img/polygon-area-trapezoid-formula.svg" width="500">

Remove the `#` before `trapezoid_signed` and `result` and replace the `???` with code to calculate the area.

In [ ]:
def area(polygon):
    result = 0

    for i in range(len(polygon) - 1):
        x_this, y_this = polygon[i]
        x_next, y_next = polygon[i + 1]
        # print(x_this, y_this, x_next, y_next)

        # trapezoid_signed = ???
        # result += ???

    return result

area(sample_shape)

The resulting area should be

```
374.8152
```

In [ ]:
nearly_circular = np.array([
    [ 2.1 ,  7.02], [ 3.22,  8.94], [ 4.88,  9.79], [ 7.57, 10.41],
    [ 9.44,  9.44], [10.47,  7.95], [10.95,  6.56], [10.54,  4.82],
    [ 9.94,  3.87], [ 8.86,  2.83], [ 7.01,  1.79], [ 4.98,  1.98],
    [ 3.41,  2.64], [ 2.62,  3.82], [ 2.02,  5.32], [ 2.1 ,  7.02]])

In [ ]:
cute_snek = np.array([
    [ 3.95,  9.15], [ 3.59, 10.39], [ 2.47, 10.96], [ 1.44, 10.51],
    [ 0.91,  9.65], [ 2.13,  9.24], [-0.04,  8.98], [ 2.2 ,  9.03],
    [ 1.11,  8.48], [ 1.54,  7.88], [ 2.37,  8.  ], [ 3.21,  7.93],
    [ 4.23,  6.47], [ 4.21,  5.21], [ 4.59,  3.08], [ 5.29,  2.03],
    [ 6.65,  0.6 ], [ 8.56,  0.69], [ 9.8 ,  2.12], [10.57,  4.2 ],
    [10.8 ,  6.64], [11.43,  7.67], [12.48,  7.62], [12.84,  6.14],
    [13.1 ,  3.53], [13.29,  1.62], [13.91,  0.5 ], [15.34,  0.33],
    [16.73,  1.31], [17.73,  2.75], [18.69,  4.8 ], [19.5 ,  6.02],
    [21.46,  7.  ], [22.34,  7.29], [20.65,  7.45], [19.26,  7.02],
    [18.02,  5.99], [17.23,  4.49], [16.2 ,  2.98], [14.72,  2.65],
    [14.1 ,  3.77], [14.25,  5.52], [13.96,  6.95], [13.43,  8.74],
    [12.05,  9.46], [10.26,  9.03], [ 9.54,  7.45], [ 9.18,  5.71],
    [ 8.58,  3.45], [ 8.03,  2.44], [ 6.86,  2.52], [ 6.02,  3.71],
    [ 5.52,  5.19], [ 5.28,  7.06], [ 4.61,  8.11], [ 3.95,  9.15]])

### Test it on more shapes

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))

ax1.plot(nearly_circular[:, 0], nearly_circular[:, 1], marker="o")
ax1.set_xlabel("x")
ax1.set_ylabel("y")

ax2.plot(cute_snek[:, 0], cute_snek[:, 1], marker="o")
ax2.set_xlabel("x")
ax2.set_ylabel("y")

None

In [ ]:
def polsby_popper_score(polygon):
    return 4*np.pi * area(polygon) / perimeter(polygon)**2

In [ ]:
polsby_popper_score(nearly_circular)

In [ ]:
polsby_popper_score(cute_snek)

## Analysis of real data

In [ ]:
district_shapes = await data.json_zip("ND-district-shapes.json.zip")

In [ ]:
fig, ax = plt.subplots()

for polygon in district_shapes:
    ax.plot([x for x, y in polygon], [y for x, y in polygon])

ax.set_xlabel("kilometers east")
ax.set_ylabel("kilometers north")

None

Compute the Polsby–Popper score for all the districts in North Dakota.

* How many are below 0.3?
* How many are between 0.3 and 0.7?
* How many are above 0.7?

Don't just count them. Use Python to count for you!

<br><br><br><br><br>

# 5. Pick a state to analyze in more detail

Either run the following cell to pick a state from a map or load this URL in your web browser:

```
https://pivarski-princeton.s3.us-east-1.amazonaws.com/us-election-analysis/legislative-district-shapes/STATE-upper-house.geojson.zip
```

replacing `STATE` with a two-letter state abbreviation (all caps).

In [ ]:
%%html

<a id="download-a-file" href="" style="display: none;"></a>

<script>
function select_a_state_onload(frame) {
  frame.contentWindow.click_on_a_state = function(state) {
    let a = document.getElementById("download-a-file");
    a.href = `https://pivarski-princeton.s3.us-east-1.amazonaws.com/us-election-analysis/legislative-district-shapes/${state}-upper-house.geojson.zip`;
    a.click();
  }
}
</script>

<iframe
  src="img/select-a-state.svg"
  width="900"
  height="600"
  style="overflow: hidden; border: none;"
  onload="select_a_state_onload(this);"
/>

Next, unzip the file (usually you can just double-click on it in the File Explorer/Finder) and open

<a href="https://geojson.io/" style="margin-left: 40px;">https://geojson.io/</a>

Drag the uncompressed GeoJSON file into the geojson.io window and explore. You can pick districts by name by selecting the "Table" tab (as opposed to "JSON").

Do the district shapes correspond with communities or population centers?

<br><br><br><br><br>

# 6. Correspondence between the GIS app and Python

Now load the same state in Python. Replace `STATE` in the following with the two-letter abbreviation (all caps).

In [ ]:
district_gdf = await data.geojson_zip("legislative-district-shapes/STATE-upper-house.geojson.zip")
district_gdf

Compute all of the perimeters and areas with the following.

How do the areas compare with what you see in geojson.io?

In [ ]:
geom = district_gdf["geometry"]

district_gdf.assign(perimeter=geom.length, area=geom.area)

Take a look at [these two squares](https://geojson.io/#data=data:application/json,%7B%22type%22%3A%22FeatureCollection%22%2C%22name%22%3A%22hundredth-of-a-degree%22%2C%22crs%22%3A%7B%22type%22%3A%22name%22%2C%22properties%22%3A%7B%22name%22%3A%22urn%3Aogc%3Adef%3Acrs%3AOGC%3A1.3%3ACRS84%22%7D%7D%2C%22features%22%3A%5B%7B%22type%22%3A%22Feature%22%2C%22properties%22%3A%7B%22name%22%3A%22Utqiagvik%2C%20Alaska%22%2C%22description%22%3A%220.01%20degree%20square%20in%20Utqiagvik%2C%20Alaska%22%7D%2C%22geometry%22%3A%7B%22type%22%3A%22Polygon%22%2C%22coordinates%22%3A%5B%5B%5B-156.7512919353568%2C71.2964592172707%5D%2C%5B-156.7412919353568%2C71.2964592172707%5D%2C%5B-156.7412919353568%2C71.30645921727069%5D%2C%5B-156.7512919353568%2C71.30645921727069%5D%2C%5B-156.7512919353568%2C71.2964592172707%5D%5D%5D%7D%7D%2C%7B%22type%22%3A%22Feature%22%2C%22properties%22%3A%7B%22name%22%3A%22Big%20Cypress%2C%20Florida%22%2C%22description%22%3A%220.01%20degree%20square%20in%20Big%20Cypress%2C%20Florida%22%7D%2C%22geometry%22%3A%7B%22type%22%3A%22Polygon%22%2C%22coordinates%22%3A%5B%5B%5B-80.99617003528675%2C26.313874265802077%5D%2C%5B-80.98617003528676%2C26.313874265802077%5D%2C%5B-80.98617003528676%2C26.323874265802075%5D%2C%5B-80.99617003528675%2C26.323874265802075%5D%2C%5B-80.99617003528675%2C26.313874265802077%5D%5D%5D%7D%7D%5D%7D), each of which is 0.01 degrees of latitude and 0.01 degrees of longitude on each side.

Notice anything?

<br><br><br><br><br>

See [spatialreference.org](https://spatialreference.org/).

After fixing that problem (we'll talk about it), calculate Polsby–Popper scores for all rows in the table using `geom` and `assign`.

<br><br><br><br><br>

# 7. Visualizing the map in Python

In [ ]:
fig, ax = plt.subplots()

district_gdf.plot("name", ax=ax)

None

A map in which the color of each area represents some data is called a "choropleth". (Fun word!)

In [ ]:
district_gdf.explore("name")

<br><br><br><br><br>

# 8. Who lives there, who votes, and how do they vote?

Download the voter file for your chosen state. It came from [redistrictingdatahub.org](https://redistrictingdatahub.org/) (which came from [L2](https://l2-data.com/), which came from the states themselves, following the [Help America Vote Act](https://www.eac.gov/about/help_america_vote_act.aspx) of 2002).

Replace `STATE` with the two-letter abbreviation (all caps) of your chosen state:

```
https://pivarski-princeton.s3.us-east-1.amazonaws.com/us-election-analysis/redistricting-data-hub-voter-files/STATE-2022-voter-file.csv.zip
```

Unzip it and look at the CSV in anything that views CSV files (Excel, Google Sheets, a text editor, spacebar/quick-view on a Mac).

[Here is a README file](https://pivarski-princeton.s3.us-east-1.amazonaws.com/us-election-analysis/redistricting-data-hub-voter-files/README.txt) that explains the meaning of each column.

Next, load it into Python. Remember to replace `STATE` with your chosen state.

In [ ]:
voterfile_df = await data.csv_zip("redistricting-data-hub-voter-files/STATE-2022-voter-file.csv.zip")
voterfile_df

In [ ]:
voterfile_df.columns

Notice that there are no shapes (`POLYGON`) in this table. To get the data onto a map, we need to know how each of the `geoid20` numbers correspond to shapes.

The U.S. Census provides a file making such a connection. Remember to replace `STATE` with your chosen state.

In [ ]:
blocks_gdf = await data.parquet("census-block-shapes/STATE-census-blocks.parquet")
blocks_gdf

For every `geoid20` in `voterfile_df`, there is one `geoid20` in `blocks_gdf`. The Pandas [merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) function merges the two tables, matching rows by `geoid20`.

In [ ]:
gdf = blocks_gdf.merge(voterfile_df, on="geoid20")
gdf

Now you can make plots of quantities like the number of registered voters per block.

_(If your web browser can't plot this one, skip to the next cell.)_

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

gdf.plot("total_reg", ax=ax, legend=True)

None

But blocks are too small and there's too many of them. The GeoPandas [dissolve](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.dissolve.html) function combines geometries by a given column, such as county. When rows are combined, we ask for them to be summed, but first we have to drop the columns that are not numbers.

In [ ]:
gdf_county = gdf.drop(columns=["geoid20", "tract", "uacode"]).dissolve("county", aggfunc="sum")
gdf_county

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

gdf_county.plot("total_reg", ax=ax, legend=True)

None

Or U.S. Census tract (regions of about 4000 people each).

In [ ]:
gdf_tract = gdf.drop(columns=["geoid20", "county", "uacode"]).dissolve("tract", aggfunc="sum")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

gdf_tract.plot("total_reg", ax=ax, legend=True)

None

Since human population is very dense in some areas and sparse in others, it's often useful to compute fractions, such as the fraction of registered voters who voted in the 2020 election.

Uncomment the `district_gdf.plot` line to see the district boundaries on the same map.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

fraction = gdf_tract["pres20_voted_all"] / gdf_tract["pres20_reg_all"]
gdf_tract.assign(fraction=fraction).plot("fraction", ax=ax, legend=True)

# district_gdf.plot(color="none", edgecolor="red", ls=":", ax=ax)

None

Now you're ready to make maps that can answer a lot of questions:

* Who voted in the 2020 presidential election? Who voted in the 2022 midterms?
* Where do Democrats live? Where do Republicans live? (Are any other parties significant in your chosen state?)
* How are ethnicities or different language-speakers distributed?
* What about estimated income?
* What about gender and age groups? Do they vote more or less, and are they registered to different parties?

Answer these questions and anything else you're curious about!

<br><br><br><br><br>

**Suggestion:** for binary questions, like "is this tract mostly Republican?", consider setting the `cmap` to a [diverging color map](https://matplotlib.org/stable/users/explain/colors/colormaps.html#diverging).

**Another suggestion:** you can turn numerical questions into categories, like "is this tract mostly Republican, mostly Democrat, or other?" by computing boolean expressions into integers:

```python
(gdf_tract["party_rep"]/gdf_tract["total_reg"] > 0.5).astype(int)
```

and adding them up in such a way that "mostly Republican = 1" lands on a different integer than "mostly Democrat = 1".

<br><br><br><br><br>

# 9. Joining data geographically

So far, we've used [dissolve](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.dissolve.html) to combine geometries that have the same value, such has `county`, `tract`, or `uacode`.

But if you want to count voters with various properties in each legislative district, you have to merge the voter file table, whose geometry is Census blocks, with the legislative districts table, which has a different geometry.

We can do that using set-operations on shapes, such as union, intersection, and difference.

In [ ]:
nearly_circular = shapely.Polygon([
    [ 2.1 ,  7.02], [ 3.22,  8.94], [ 4.88,  9.79], [ 7.57, 10.41],
    [ 9.44,  9.44], [10.47,  7.95], [10.95,  6.56], [10.54,  4.82],
    [ 9.94,  3.87], [ 8.86,  2.83], [ 7.01,  1.79], [ 4.98,  1.98],
    [ 3.41,  2.64], [ 2.62,  3.82], [ 2.02,  5.32], [ 2.1 ,  7.02]])

In [ ]:
cute_snek = shapely.Polygon([
    [ 3.95,  9.15], [ 3.59, 10.39], [ 2.47, 10.96], [ 1.44, 10.51],
    [ 0.91,  9.65], [ 2.13,  9.24], [-0.04,  8.98], [ 2.2 ,  9.03],
    [ 1.11,  8.48], [ 1.54,  7.88], [ 2.37,  8.  ], [ 3.21,  7.93],
    [ 4.23,  6.47], [ 4.21,  5.21], [ 4.59,  3.08], [ 5.29,  2.03],
    [ 6.65,  0.6 ], [ 8.56,  0.69], [ 9.8 ,  2.12], [10.57,  4.2 ],
    [10.8 ,  6.64], [11.43,  7.67], [12.48,  7.62], [12.84,  6.14],
    [13.1 ,  3.53], [13.29,  1.62], [13.91,  0.5 ], [15.34,  0.33],
    [16.73,  1.31], [17.73,  2.75], [18.69,  4.8 ], [19.5 ,  6.02],
    [21.46,  7.  ], [22.34,  7.29], [20.65,  7.45], [19.26,  7.02],
    [18.02,  5.99], [17.23,  4.49], [16.2 ,  2.98], [14.72,  2.65],
    [14.1 ,  3.77], [14.25,  5.52], [13.96,  6.95], [13.43,  8.74],
    [12.05,  9.46], [10.26,  9.03], [ 9.54,  7.45], [ 9.18,  5.71],
    [ 8.58,  3.45], [ 8.03,  2.44], [ 6.86,  2.52], [ 6.02,  3.71],
    [ 5.52,  5.19], [ 5.28,  7.06], [ 4.61,  8.11], [ 3.95,  9.15]])

In [ ]:
nearly_circular

In [ ]:
cute_snek

In [ ]:
nearly_circular.union(cute_snek)

In [ ]:
nearly_circular.intersection(cute_snek)

In [ ]:
nearly_circular.difference(cute_snek)

In [ ]:
cute_snek.difference(nearly_circular)

In [ ]:
cute_snek.within(nearly_circular)

In [ ]:
nearly_circular.contains(cute_snek)

The GeoPandas [sjoin](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.sjoin.html) function merges GeoDataFrames by overlapping geometries.

[EPSG:4326](https://epsg.io/4326) is the coordinate system of latitude and longitude, which is fine for computing overlaps.

In [ ]:
joined_gdf = gdf.to_crs("EPSG:4326").sjoin(district_gdf.to_crs("EPSG:4326"), predicate="within")
joined_gdf

It gives us a table in which each Census block is labeled by the district `name`. To get a table in which each row is a district, [groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) name and sum (dropping non-numerical columns).

In [ ]:
df = joined_gdf.drop(columns=["geoid20", "tract", "uacode", "county", "geometry"]).groupby("name").sum()
df

The geometry is gone, but we can put it back in by a [merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) on the district name.

In [ ]:
district_all_data = district_gdf.merge(df, on="name")

Now you can make choropleths in which each solid color is a whole district.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

fraction = district_all_data["party_rep"] / district_all_data["total_reg"]
district_all_data.assign(fraction=fraction).plot("fraction", cmap="Reds", vmin=0, vmax=1, ax=ax, legend=True)

None

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

fraction = district_all_data["party_dem"] / district_all_data["total_reg"]
district_all_data.assign(fraction=fraction).plot("fraction", cmap="Blues", vmin=0, vmax=1, ax=ax, legend=True)

None

Compare per-district maps with per-population (Census tract) maps. Do you see any evidence of gerrymandering?
* Do any districts "pack" similar voters into one district, so that fewer contribute to surrounding districts?
* Do any districts "crack" a group of similar voters who would be a majority in one district into a minority in many districts?

Use everything you've learned!

<br><br><br><br><br>

**Bonus:** you can do the same for your state's lower house (if it has two legislative houses) or for its representatives in the national U.S. Congress (if it has multiple representatives).

In [ ]:
lower_house_gdf = await data.geojson_zip("legislative-district-shapes/STATE-lower-house.geojson.zip")
lower_house_gdf

In [ ]:
congressional_gdf = await data.geojson_zip("legislative-district-shapes/STATE-congressional.geojson.zip")
congressional_gdf

<br><br><br><br><br>

**More bonus:** Can you draw lines that are more fair?

<a href="https://districtr.org/"><img src="/2025-07-12-scipy-teen-track/img/districtr-logo.jpg" width="500"></a>